# Лабораторная работа 1. Инструменты, данные и первый ориентир

**Курс «Машинное обучение», 4 курс, каф. ФН1**

| | |
|---|---|
| Место в курсе | первое занятие курса, **до** лекции 1 |
| Опора на лекции | нет — работа опирается только на линейную алгебру и матанализ |
| Трудоёмкость | 2 ч аудиторно + домашняя работа (3–4 ч) |

## Цель работы

Пройти путь от сырой таблицы до матрицы «объекты–признаки», пригодной для обучения, и построить ориентир, с которым дальше сравниваются все модели курса. Попутно освоить векторизацию в NumPy: весь курс записан в матричной форме, и она понадобится на каждом занятии.

## Как устроено занятие

Ноутбук разбирается в аудитории: код запускается и обсуждается по ходу.
Большая часть ячеек уже написана — их нужно **прочитать и запустить**,
разобравшись, что происходит и почему.

Ячейки, помеченные `# ✍ ЗАДАНИЕ НА СЕМИНАРЕ`, заполняются самостоятельно
прямо на занятии; их немного, и каждая занимает несколько строк. Ячейки
**Вывод** — тоже ваши: короткий ответ своими словами на поставленный вопрос.

Дома выполняется отдельный ноутбук `lab01_homework.ipynb` — там
задания крупнее и делать их нужно самому.

> **Индивидуальный вариант.** Датасет и набор методов выдаются по вашему ФИО
> (см. ячейку ниже) — и на семинаре, и в домашней работе.

## Как устроен практикум

Девять занятий, чередующихся с лекциями, причём курс начинается практикумом:

```
Зан. 1 → Лек 1 → Зан. 2 → Лек 2 → … → Лек 8 → Зан. 9
```

Значит, занятие $k+1$ закрепляет лекцию $k$. Сегодняшнее идёт до первой лекции
и теории курса не требует — это подготовка инструментов и данных.

In [ ]:
# Служебная ячейка: импорты, стиль графиков, воспроизводимость.
import sys, pathlib, warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

# Модули практикума (variants.py, labdata.py) ищем рядом с ноутбуком.
# Если их нет -- значит, ноутбук открыт в Colab: скачиваем из репозитория курса.
COURSE_FILES_URL = "https://raw.githubusercontent.com/sharipovaka/mltest1/main/notebooks"


def _course_modules_dir():
    here = pathlib.Path.cwd()
    for parent in [here, *here.parents][:4]:
        if (parent / "variants.py").exists():
            return str(parent)
    import urllib.request
    for name in ("variants.py", "labdata.py"):
        if not pathlib.Path(name).exists():
            urllib.request.urlretrieve(f"{COURSE_FILES_URL}/{name}", name)
            print(f"загружен {name} из репозитория курса")
    return str(here)


sys.path.insert(0, _course_modules_dir())
from variants import get_variant, describe_variant  # noqa: E402

RANDOM_STATE = 42          # единый seed на всю работу: результаты воспроизводимы
rng = np.random.default_rng(RANDOM_STATE)

plt.rcParams.update({
    "figure.figsize": (7.5, 4.5),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

print("numpy", np.__version__, "| pandas", pd.__version__)

from variants import make_table

## Индивидуальный вариант

Впишите своё ФИО (или почту) в переменную `STUDENT` — вариант вычисляется детерминированно, при повторном запуске он тот же самый.

In [ ]:
STUDENT = "Фамилия Имя Отчество"   # <-- впишите себя

variant = get_variant(STUDENT, lab=1)
describe_variant(variant)

---
# Часть 1. Векторизация вместо циклов

Курс записан в матричных обозначениях: матрица «объекты–признаки» размера
$\ell\times n$ — это массив NumPy формы `(l, n)`. Правило, которое сэкономит
десятки часов: **если вы пишете цикл по объектам выборки, почти наверняка есть
векторизованная запись — короче и в десятки раз быстрее.**

Понадобится матрица попарных квадратов расстояний
$D_{ij} = \|a_i - b_j\|^2$. Считать её двойным циклом можно, но не нужно:

$$
\|a - b\|^2 = \|a\|^2 - 2\langle a, b\rangle + \|b\|^2 .
$$

In [ ]:
# ✍ ЗАДАНИЕ НА СЕМИНАРЕ — допишите эту ячейку

def dists_vectorized(A, B):
    """Попарные квадраты расстояний без единого цикла.

    Подсказка: np.sum(A**2, axis=1)[:, None] -- столбец квадратов норм,
    A @ B.T -- матрица скалярных произведений, сложить поможет broadcasting.
    """
    # TODO (3-4 строки). В конце верните np.maximum(D, 0.0) -- см. вывод ниже.
    raise NotImplementedError

In [ ]:
import time


def dists_loops(A, B):
    """Тот же результат двойным циклом -- эталон корректности."""
    D = np.empty((A.shape[0], B.shape[0]))
    for i in range(A.shape[0]):
        for j in range(B.shape[0]):
            diff = A[i] - B[j]
            D[i, j] = diff @ diff
    return D


A, B = rng.normal(size=(400, 20)), rng.normal(size=(300, 20))
t0 = time.perf_counter(); D_loop = dists_loops(A, B); t_loop = time.perf_counter() - t0
t0 = time.perf_counter(); D_vec = dists_vectorized(A, B); t_vec = time.perf_counter() - t0

print(f"максимальное расхождение: {np.abs(D_loop - D_vec).max():.2e}")
print(f"цикл {t_loop * 1000:.0f} мс, векторизация {t_vec * 1000:.0f} мс "
      f"-> ускорение в {t_loop / t_vec:.0f} раз")

> **Вывод.** Во сколько раз быстрее? Почему расхождение не строго нулевое и зачем `np.maximum(D, 0)`?
>
> *(ваш ответ здесь)*

---
# Часть 2. Что лежит в таблице

Данные индивидуальные и сознательно «грязные» — примерно такой вид имеет
выгрузка из реальной информационной системы.

In [ ]:
df_raw, meta = make_table(variant)
TASK, TARGET = meta["task"], meta["target"]

print(f"{meta['domain']}: {df_raw.shape[0]} объектов, {df_raw.shape[1] - 1} признаков")
print(f"тип задачи: {TASK}, целевая переменная: {TARGET}")
df_raw.head(5)

In [ ]:
# Сводка по столбцам: тип, число уникальных значений, пропуски
info = pd.DataFrame({
    "тип": df_raw.dtypes.astype(str),
    "уникальных": df_raw.nunique(),
    "доля пропусков, %": (df_raw.isna().mean() * 100).round(1),
    "пример": df_raw.apply(lambda s: s.dropna().iloc[0] if s.notna().any() else None),
})
display(info)

> **Вывод.** Какие столбцы прочитаны как `object`, хотя по смыслу числовые? Какие бесполезны уже сейчас?
>
> *(ваш ответ здесь)*

---
# Часть 3. Приводим в порядок

Тип признака определяет, что с ним вообще можно делать:

| Тип | Множество значений | Пример | Что осмысленно |
|---|---|---|---|
| бинарный | $\{0,1\}$ | «есть балкон» | всё |
| номинальный | конечное, без порядка | «район» | сравнение на равенство |
| порядковый | конечное, упорядоченное | «ремонт: без / косм. / евро» | сравнение $\le$ |
| количественный | $\mathbb{R}$ | «площадь» | арифметика |

Ошибка здесь стоит дорого: закодировав «район» числами 1..5, мы сообщим модели,
что «Юг» больше «Севера», а «Запад» — ровно посередине между ними.

In [ ]:
# ✍ ЗАДАНИЕ НА СЕМИНАРЕ — допишите эту ячейку

def to_numeric_safe(s: pd.Series, threshold: float = 0.2) -> pd.Series:
    """Строковый столбец -> числовой, если он на самом деле числовой."""
    if s.dtype != object:
        return s

    # TODO (3-4 строки):
    #   1) убрать пробелы (в т.ч. неразрывные) и заменить ',' на '.';
    #   2) pd.to_numeric(..., errors="coerce");
    #   3) если доля непреобразованных НЕПУСТЫХ значений > threshold,
    #      столбец действительно нечисловой -- вернуть s без изменений.
    raise NotImplementedError

In [ ]:
df = df_raw.copy()
for col in df.columns:
    before = df[col].dtype
    df[col] = to_numeric_safe(df[col])
    if before != df[col].dtype:
        print(f"{col}: {before} -> {df[col].dtype}")

# Категории приходят в разном написании: «Москва», «москва », «МОСКВА»
cat_cols = meta["categorical"]
print("\nбыло категорий:", {c: df[c].nunique() for c in cat_cols})
for col in cat_cols:
    df[col] = df[col].astype(object).map(
        lambda v: v.strip().lower() if isinstance(v, str) else v)
print("стало категорий:", {c: df[c].nunique() for c in cat_cols})

In [ ]:
# Дубликаты строк и вырожденные признаки
print(f"полных дубликатов: {df.duplicated().sum()}")
df = df.drop_duplicates().reset_index(drop=True)

drop = [c for c in df.columns.drop(TARGET)
        if df[c].nunique(dropna=False) <= 1
        or df[c].value_counts(normalize=True, dropna=False).iloc[0] > 0.99]
print(f"вырожденные признаки (удаляем): {drop}")
df = df.drop(columns=drop)
print(f"осталось: {len(df)} объектов, {df.shape[1] - 1} признаков")

### Выбросы и опечатки

Два рабочих правила поиска выбросов: $|x - \overline x| > 3\sigma$ (страдает от
самих выбросов — они раздувают $\sigma$) и межквартильное
$x \notin [Q_1 - 1.5\,\mathrm{IQR},\ Q_3 + 1.5\,\mathrm{IQR}]$ (устойчивое).

Среди выбросов есть особая группа: значения, превышающие 99-й перцентиль
примерно в 100 раз. Это не редкие объекты, а **потерянный десятичный
разделитель** — их надо чинить, а не удалять.

> **Напоминание — квантиль, квартиль, IQR.** Квантиль уровня $q$ — это значение $x_q$, ниже которого лежит доля $q$
> наблюдений: $x_{0.5}$ — медиана, $x_{0.99}$ — 99-й перцентиль. Квартили — это
> квантили уровней $0.25$, $0.5$, $0.75$; обозначаются $Q_1, Q_2, Q_3$.
> Межквартильный размах $\mathrm{IQR} = Q_3 - Q_1$ — ширина «средней половины»
> данных. Он устойчив к выбросам: сколько бы ни было экстремальных значений,
> пока их меньше четверти, границы $Q_1$ и $Q_3$ не сдвинутся. У $\overline x$
> и $\sigma$ такого свойства нет — один выброс тянет обе величины за собой.
> В NumPy: `np.quantile(x, [0.25, 0.75])`.

In [ ]:
num_cols = df.select_dtypes(include="number").columns.drop(TARGET)
report = []
for col in num_cols:
    s = df[col].dropna()
    q1, q3 = s.quantile([0.25, 0.75]); iqr = q3 - q1
    report.append({"признак": col,
                   "IQR-правило": int(((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).sum()),
                   "3 sigma": int((np.abs(s - s.mean()) > 3 * s.std()).sum()),
                   "макс / q_0.99": round(s.max() / s.quantile(0.99), 1)})
display(pd.DataFrame(report).set_index("признак"))

In [ ]:
# Чиним опечатки масштаба и смотрим, как меняется связь признака с целью
for col in num_cols:
    mask = df[col] > 30 * df[col].quantile(0.99)
    if mask.sum():
        before = df[[col, TARGET]].corr().iloc[0, 1]
        df.loc[mask, col] = df.loc[mask, col] / 100.0
        print(f"{col}: исправлено {mask.sum()} значений, "
              f"корреляция с целью {before:+.3f} -> {df[[col, TARGET]].corr().iloc[0, 1]:+.3f}")

> **Вывод.** Какое правило нашло больше выбросов и почему? Что произошло с корреляцией после исправления опечаток?
>
> *(ваш ответ здесь)*

---
# Часть 4. Признак, которого не должно быть

В таблице есть столбец `id`. Формально это число, и его можно подать в модель.
Посмотрим, что будет.

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

model, scoring = ((DecisionTreeRegressor(max_depth=4, random_state=RANDOM_STATE), "r2")
                  if TASK == "regression"
                  else (DecisionTreeClassifier(max_depth=4, random_state=RANDOM_STATE), "roc_auc"))
score = cross_val_score(model, df[["id"]], df[TARGET], cv=5, scoring=scoring)
print(f"качество ({scoring}) по ОДНОМУ признаку id: {score.mean():.3f}")

In [ ]:
fig, ax = plt.subplots()
ax.scatter(df["id"], df[TARGET], s=6, alpha=0.35, color="#C97A2B")
ax.set_xlabel("id"); ax.set_ylabel(TARGET)
ax.set_title("Целевая переменная против идентификатора")
plt.tight_layout(); plt.show()

df = df.drop(columns=["id"])
print("столбец id удалён")

> **Вывод.** Откуда взялась такая связь и почему `id` необходимо удалить, хотя он улучшает качество?
>
> *(ваш ответ здесь)*

---
# Часть 5. Разбиение и предобработка без утечек

Главное правило всего курса:

> Все параметры предобработки — средние, дисперсии, медианы для заполнения
> пропусков, список категорий — вычисляются **только по обучающей части**
> и затем применяются к контрольной.

Иначе информация о контрольной выборке просачивается в обучение, и оценка
качества оказывается завышенной. Насколько именно — измерим на занятии 5.

Технически это делает `Pipeline` с `ColumnTransformer`: они запоминают
параметры на `fit` и применяют их в `transform`.

> **Напоминание — стратификация.** При случайном делении выборки доли классов в частях могут заметно разойтись —
> особенно если один класс редкий. Стратифицированное разбиение делит выборку
> внутри каждого класса отдельно, поэтому доли сохраняются. В `sklearn` это
> аргумент `stratify=y`. Для регрессии он не нужен (классов нет), а для
> классификации — практически всегда: иначе часть разброса оценки качества
> объясняется просто неудачным делением.

In [ ]:
from sklearn.model_selection import train_test_split

X, y = df.drop(columns=[TARGET]), df[TARGET]
num_cols_final = list(X.select_dtypes(include="number").columns)
cat_cols_final = [c for c in X.columns if c not in num_cols_final]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE,
    stratify=y if TASK == "classification" else None)
print(f"обучающая {X_train.shape[0]}, контрольная {X_test.shape[0]}")
print(f"числовых {len(num_cols_final)}, категориальных {len(cat_cols_final)}")

In [ ]:
# ✍ ЗАДАНИЕ НА СЕМИНАРЕ — допишите эту ячейку

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# TODO: соберите ColumnTransformer из двух веток:
#   числовые (num_cols_final)  -> SimpleImputer(strategy="median") + StandardScaler
#   категории (cat_cols_final) -> SimpleImputer(strategy="most_frequent")
#                                 + OneHotEncoder(handle_unknown="ignore",
#                                                 sparse_output=False)
preprocessor = ...

In [ ]:
X_train_t = preprocessor.fit_transform(X_train)   # fit ТОЛЬКО на обучающей части
X_test_t = preprocessor.transform(X_test)         # на контрольной -- только transform
feature_names = preprocessor.get_feature_names_out()

print(f"матрица объектов-признаков: {X_train_t.shape} (было {X_train.shape[1]} столбцов)")
print("первые признаки:", list(feature_names[:5]))

---
# Часть 6. Ориентир: с чем сравнивать модели

Прежде чем радоваться качеству, надо знать, сколько даёт **тривиальный ответ** —
константа. Для регрессии это среднее, для классификации — самый частый класс.

Строгие определения функции потерь и риска будут на лекции 1; пока пользуемся
привычными:

$$
\mathrm{MSE} = \frac1\ell\sum_i (a(x_i) - y_i)^2, \qquad
R^2 = 1 - \frac{\sum_i (a(x_i) - y_i)^2}{\sum_i (\overline y - y_i)^2}.
$$

Заметьте: $R^2 = 0$ — это ровно качество константного прогноза средним.

In [ ]:
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import accuracy_score, r2_score, roc_auc_score

if TASK == "regression":
    pairs = [("константа (среднее)", DummyRegressor(strategy="mean")),
             ("линейная регрессия", LinearRegression())]
    score = lambda m: {"R2": r2_score(y_test, m.predict(X_test_t))}
else:
    pairs = [("константа (частый класс)", DummyClassifier(strategy="most_frequent")),
             ("логистическая регрессия", LogisticRegression(max_iter=2000))]
    score = lambda m: {"accuracy": accuracy_score(y_test, m.predict(X_test_t)),
                       "ROC-AUC": roc_auc_score(y_test, m.predict_proba(X_test_t)[:, 1])}

results = {name: score(m.fit(X_train_t, y_train)) for name, m in pairs}
display(pd.DataFrame(results).T.round(4))

> **Вывод.** Насколько модель лучше константы? Что означает `accuracy = 0.8` у константного классификатора при несбалансированных классах?
>
> *(ваш ответ здесь)*

In [ ]:
# Сохраняем подготовленные данные: понадобятся на следующих занятиях
import pathlib, joblib

data_dir = pathlib.Path("data"); data_dir.mkdir(exist_ok=True)
np.savez_compressed(data_dir / "prepared.npz", X_train=X_train_t, X_test=X_test_t,
                    y_train=np.asarray(y_train), y_test=np.asarray(y_test),
                    feature_names=np.asarray(feature_names, dtype=object))
joblib.dump(preprocessor, data_dir / "preprocessor.joblib")
print("сохранено:", *[p.name for p in sorted(data_dir.iterdir())])

## Итоги занятия

Ответьте письменно на контрольные вопросы (по 2–4 предложения):

1. Чем номинальный признак отличается от порядкового и почему их нельзя кодировать одинаково? Приведите пример из своей таблицы.
2. Что такое утечка данных? Приведите два примера: один из вашей таблицы, второй — придуманный для задачи «вернёт ли клиент кредит».
3. Почему параметры масштабирования оценивают только по обучающей выборке? Что именно завышается, если это правило нарушить?
4. Вы получили accuracy 0.93. Какие ещё два числа нужно знать, чтобы понять, хороший это результат?

---

**Дома:** откройте `lab01_homework.ipynb` — там две задачи: механизм пропусков и своя реализация One-Hot.